# Kannada-MNIST

## EDA

Explore handwritten Kannada digits represented as 28 × 28 grayscale images, with ten labels (0–9).

### Competition links

- [Overview](https://www.kaggle.com/competitions/Kannada-MNIST)
- [Data](https://www.kaggle.com/competitions/Kannada-MNIST/data)
- [Evaluation](https://www.kaggle.com/competitions/Kannada-MNIST/overview/evaluation)
- [Rules](https://www.kaggle.com/competitions/Kannada-MNIST/rules)
- [Original dataset and research](https://github.com/vinayprabhu/Kannada_MNIST)

### Project scope

1. Confirm the selected Python/GPU environment.
2. Resolve data with KaggleHub and inspect file schemas.
3. Compare label distributions and representative images from the training and Dig-MNIST datasets.
4. Prepare a separate CNN baseline after inspecting the data.

This is a historical Kaggle **code competition**, evaluated by classification accuracy. A notebook upload and a scored competition submission are separate actions. The starter metadata is for private exploratory execution, not a final submission configuration.

### Data and evaluation plan

| File | Role |
| --- | --- |
| `train.csv` | Labeled images for model development and a fixed stratified validation split |
| `test.csv` | Unlabeled competition images with an `id` column |
| `sample_submission.csv` | Required `id,label` column order and image IDs |
| `Dig-MNIST.csv` | Separate labeled challenge set, initially held out from training |

Keep Dig-MNIST scores separate from in-distribution validation scores. Do not tune a model on an evaluation set and then report that same set as an untouched test.

Use the Kaggle GPU Dev Container's `/usr/bin/python3` locally. On a remote Kaggle session, select its Python kernel and verify the environment again. This notebook does not train a model or submit predictions.


In [ ]:
import os
import sys
import random
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python:", sys.version)
print("Interpreter:", sys.executable)
print("Host:", os.uname().nodename)
print(
    {
        name: version(name)
        for name in ["torch", "numpy", "pandas", "plotly", "kagglehub"]
    }
)

assert torch.cuda.is_available(), "Select a GPU kernel and verify CUDA access."
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
probe = torch.arange(16, dtype=torch.float32, device=device)
assert (probe @ probe).item() == 1240.0
print("CUDA computation passed")

pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"


## Data access

Reuse local competition CSVs when present. Otherwise, download through `kagglehub.competition_download()` and use its returned path. On Kaggle, the requested local download directory may be ignored.

Local authentication comes from the repository-root `.env` via `load_dotenv(override=False)`. Existing environment variables take precedence. Remote kernels use their own session authentication and filesystem. Never print tokens or connection URLs.


In [ ]:
import kagglehub
from dotenv import load_dotenv

COMPETITION = "Kannada-MNIST"
repo_dir = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "competitions" / COMPETITION).is_dir()
    ),
    None,
)
project_dir = (
    repo_dir / "competitions" / COMPETITION
    if repo_dir is not None
    else Path.cwd() if Path.cwd().name == COMPETITION else Path.cwd() / COMPETITION
)
output_dir = project_dir / "output"

if repo_dir is not None:
    load_dotenv(repo_dir / ".env", override=False)

required_files = ("train.csv", "test.csv", "sample_submission.csv", "Dig-MNIST.csv")
data_dir = project_dir / "data"
if not all((data_dir / name).is_file() for name in required_files):
    data_dir = Path(
        kagglehub.competition_download(
            COMPETITION,
            output_dir=str(data_dir.resolve()),
        )
    )

missing = [name for name in required_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing competition files in {data_dir}: {missing}")

print("Competition data:", data_dir.resolve())
print("Output directory:", output_dir.resolve())


## File schemas and integrity

Read pixel values as unsigned bytes. Inspect the submission template rather than reusing Digit Recognizer's `ImageId` convention. Row counts below describe the files available in this execution environment; a code-competition scoring run can use a different test set.


In [ ]:
train = pd.read_csv(data_dir / "train.csv", dtype=np.int16)
dig_mnist = pd.read_csv(data_dir / "Dig-MNIST.csv", dtype=np.int16)
submission_template = pd.read_csv(data_dir / "sample_submission.csv")
test = pd.read_csv(data_dir / "test.csv")

pixels = [f"pixel{i}" for i in range(784)]
assert train.columns.tolist() == ["label", *pixels]
assert dig_mnist.columns.tolist() == ["label", *pixels]
assert submission_template.columns.tolist() == ["id", "label"]
assert test.columns.tolist() == ["id", *pixels]
assert len(test) == len(submission_template)
assert test["id"].is_unique
assert test["id"].equals(submission_template["id"])

for name, frame in [("train", train), ("Dig-MNIST", dig_mnist), ("test", test)]:
    assert not frame.isna().any().any(), f"Missing values in {name}"
    assert frame[pixels].min().min() >= 0 and frame[pixels].max().max() <= 255
    if "label" in frame:
        assert frame["label"].between(0, 9).all()

display(
    pd.DataFrame(
        [
            {"file": "train.csv", "rows": len(train), "columns": len(train.columns)},
            {
                "file": "Dig-MNIST.csv",
                "rows": len(dig_mnist),
                "columns": len(dig_mnist.columns),
            },
            {"file": "test.csv", "rows": len(test), "columns": len(test.columns)},
            {
                "file": "sample_submission.csv",
                "rows": len(submission_template),
                "columns": len(submission_template.columns),
            },
        ]
    )
)
display(train.head())
display(submission_template.head())


## Label distributions

Compare training and Dig-MNIST class counts. Similar class balance does not imply identical handwriting styles or image quality.


In [ ]:
counts = pd.concat(
    [
        frame["label"]
        .value_counts()
        .reindex(range(10), fill_value=0)
        .rename_axis("digit")
        .reset_index(name="samples")
        .assign(dataset=name)
        for name, frame in [("Train", train), ("Dig-MNIST", dig_mnist)]
    ],
    ignore_index=True,
)
counts["digit"] = counts["digit"].astype(str)
fig = px.bar(
    counts,
    x="digit",
    y="samples",
    color="dataset",
    barmode="group",
    labels={"digit": "Digit label", "samples": "Images", "dataset": "Dataset"},
    title="Label counts: training data and Dig-MNIST",
)
fig.show()


## Representative images

Each image includes its numeric label and Kannada Unicode numeral directly above it, within the same interactive figure. The five-column, two-row layout follows digits 0–4 and 5–9.

Compare the reference glyph with the handwritten image below it; handwritten shapes may vary. Both datasets use the same label/reference layout, with one seeded random sample per digit.


In [ ]:
def show_digit_examples(frame, title):
    rng = np.random.default_rng(SEED)
    fig = make_subplots(
        rows=2,
        cols=5,
        subplot_titles=[f"<b>{digit}</b> — {chr(0x0CE6 + digit)}" for digit in range(10)],
        vertical_spacing=0.22,
    )
    for digit in range(10):
        candidates = frame.index[frame["label"] == digit].to_numpy()
        if len(candidates) == 0:
            continue
        index = rng.choice(candidates)
        image = (
            frame.loc[index, pixels].to_numpy(dtype=np.float32).reshape(28, 28) / 255.0
        )
        fig.add_trace(
            go.Heatmap(z=image, colorscale="Gray", zmin=0, zmax=1, showscale=False),
            row=digit // 5 + 1,
            col=digit % 5 + 1,
        )
        axis = "x" if digit == 0 else f"x{digit + 1}"
        fig.update_yaxes(
            autorange="reversed",
            visible=False,
            scaleanchor=axis,
            scaleratio=1,
            row=digit // 5 + 1,
            col=digit % 5 + 1,
        )
    fig.update_xaxes(visible=False, constrain="domain")
    fig.update_annotations(font=dict(size=20))
    fig.update_layout(title=title, height=600, margin=dict(t=100))
    fig.show()


show_digit_examples(train, "Training examples by digit")
show_digit_examples(dig_mnist, "Dig-MNIST examples by digit")


## Next steps

- Create `baseline.ipynb` with a fixed stratified train/validation split and an initial CNN.
- Track training/validation loss, accuracy, and overall/per-digit precision, recall, and F1.
- Evaluate Dig-MNIST separately before considering how to use it during development.
- Preserve `sample_submission.csv` IDs and column order when generating predictions; do not hard-code the number of test rows.
- Recheck current competition submission rules before preparing a scoring notebook. This EDA configuration is not a submission pipeline.
- Save local outputs and export an HTML report only when the exploration is ready to publish.
